# MNIST Model Stealing — D1, D2, and DisGUIDE

End-to-end notebook for stealing an **undefended** MNIST LeNet victim with three data-free attacks:

| Attack | Kind | Description |
|--------|------|-------------|
| **D1** | `latent_manifold` | CEM over a convolutional generator latent space |
| **D2** | `procedural_natural` | CEM over procedural Fourier/grid/blob parameters |
| **DisGUIDE** | `disguide` | GAN-style generator + substitute ensemble with disagreement |

## Pipeline

1. Download MNIST
2. Train the victim LeNet (no prediction or query defense)
3. Run each attack, build a transfer set from victim soft labels, and train a substitute LeNet

Outputs are written under `runs/notebook/mnist_stealing/`.

**Run the setup cell first** (it adds the repository root to `sys.path` so `defenses` imports work from any notebook working directory).

In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "flowguard_notebook_utils.py").exists():
    UTILS_DIR = NOTEBOOK_DIR
else:
    UTILS_DIR = NOTEBOOK_DIR / "notebooks" / "flow_matching_defenses"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

from flowguard_notebook_utils import apply_nature_style, setup_notebook_environment

PROJECT_ROOT, SRC_ROOT, UTILS_DIR = setup_notebook_environment(NOTEBOOK_DIR)

from defenses import datasets as legacy_datasets
from flowguard.experiments.factory import build_experiment_spec
from flowguard.experiments.spec import AttackKind
from flowguard.orchestration import run_experiment
from flowguard.training.target import train_target_model

apply_nature_style()

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

DATASET = "MNIST"
ARCHITECTURE = "lenet"
RUN_ROOT = PROJECT_ROOT / "runs" / "notebook" / "mnist_stealing"
TARGET_DIR = RUN_ROOT / "victim_lenet_nodefense" / "target_model"
FIGURE_DIR = RUN_ROOT / "figures"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Tuned best-guess budgets targeting ~90% substitute accuracy on MNIST.
# Data-free attacks need far more queries than the 512 smoke setting (~10% acc).
# Reference: supervised transfer-set stealing reaches ~99% at 60k queries on MNIST.
VICTIM_EPOCHS = 10
SUBSTITUTE_EPOCHS = 30
QUERY_BUDGET = 60_000
ATTACK_BATCH_SIZE = 1
CEM_POPULATION = 32
CEM_ELITE_FRACTION = 0.25
CEM_LATENT_DIM = 128
DISGUIDE_BATCH_SIZE = 32
NUM_WORKERS = 0  # required for Jupyter on macOS (avoid DataLoader multiprocessing errors)
REUSE_ATTACK_RUNS = False  # set True to skip attacks whose output dirs already exist

MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Run root: {RUN_ROOT}")

## 1 — Download MNIST

We use the repository's legacy MNIST wrapper (normalized tensors, 1×28×28).

In [ ]:
def to_mnist_image_array(tensor: torch.Tensor | np.ndarray) -> np.ndarray:
    """Map a query tensor to [0, 1] for plotting (normalized MNIST or tanh DisGUIDE)."""
    arr = np.asarray(tensor, dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[0]
    if arr.min() < -0.05:
        return np.clip((arr + 1.0) / 2.0, 0.0, 1.0)
    return np.clip(arr * MNIST_STD + MNIST_MEAN, 0.0, 1.0)


def load_mnist_splits(*, download: bool = True):
    family = legacy_datasets.dataset_to_modelfamily[DATASET]
    train_tf = legacy_datasets.modelfamily_to_transforms[family]["train"]
    test_tf = legacy_datasets.modelfamily_to_transforms[family]["test"]
    trainset = legacy_datasets.MNIST(train=True, transform=train_tf, download=download)
    testset = legacy_datasets.MNIST(train=False, transform=test_tf, download=download)
    return trainset, testset


trainset, testset = load_mnist_splits(download=True)
print(f"Train samples: {len(trainset):,}")
print(f"Test samples:  {len(testset):,}")
print(f"Classes:       {trainset.classes}")

fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for axis, index in zip(axes, range(8), strict=True):
    image, label = trainset[index]
    axis.imshow(to_mnist_image_array(image), cmap="gray", vmin=0.0, vmax=1.0)
    axis.set_title(str(label), fontsize=8)
    axis.axis("off")
fig.suptitle("MNIST training samples", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mnist_samples.pdf", bbox_inches="tight")
plt.show()

## 2 — Train the undefended victim LeNet

This checkpoint is the black-box target for all attacks. No prediction defense and no query defense (`noop`).

In [ ]:
def build_attack_spec(
    *,
    name: str,
    attack_kind: AttackKind,
    attack_mode: str,
    attack_extra: dict,
    attack_batch_size: int = ATTACK_BATCH_SIZE,
    substitute_architecture: str = ARCHITECTURE,
) -> object:
    spec = build_experiment_spec(
        name=name,
        dataset=DATASET,
        target_architecture=ARCHITECTURE,
        substitute_architecture=substitute_architecture,
        attack_kind=attack_kind,
        attack_mode=attack_mode,
        query_dataset=DATASET,
        query_budget=QUERY_BUDGET,
        target_checkpoint_dir=str(TARGET_DIR),
        target_device=DEVICE,
        substitute_device=DEVICE,
        prediction_defense="none",
        query_defense="noop",
        attack_batch_size=attack_batch_size,
        attack_extra=attack_extra,
        training_batch_size=128,
        epochs=SUBSTITUTE_EPOCHS,
        query_download=True,
        dataset_download=True,
        verbose=True,
    )
    spec.training.num_workers = NUM_WORKERS
    return spec


victim_spec = build_attack_spec(
    name="mnist-victim-lenet",
    attack_kind=AttackKind.TRANSFER_SET,
    attack_mode="naive",
    attack_extra={},
)
victim_spec.training.epochs = VICTIM_EPOCHS

if TARGET_DIR.exists() and any(TARGET_DIR.glob("checkpoint*.pth.tar")):
    print(f"Reusing existing victim checkpoint at {TARGET_DIR}")
else:
    print(f"Training victim LeNet for {VICTIM_EPOCHS} epochs ...")
    train_target_model(victim_spec, output_dir=TARGET_DIR)
    print(f"Victim checkpoint saved to {TARGET_DIR}")

## 3 — Steal the victim with D1, D2, and DisGUIDE

Each `run_experiment` call:

1. Queries the undefended victim to build a transfer set
2. Trains a substitute LeNet on that transfer set
3. Reports substitute **accuracy** (on MNIST test) and **fidelity** (agreement with the victim)

**D1/D2** use CEM on native 28×28 MNIST tensors and **train a substitute inline** on each elite batch (like DisGUIDE/MAZE), with `metrics.csv` progress.

**DisGUIDE** follows `disguide-main`: `GeneratorA` + `tanh` on **32×32** inputs and a **LeNet5** substitute ensemble (the legacy 28×28 LeNet breaks on 32×32 tensors). The victim stays 28×28 LeNet; queries are resized before the black-box call.

Defaults: **60k queries**, **150 inner iters/epoch** for DisGUIDE, **30 substitute epochs**. Expect long runtimes on GPU/MPS.

In [ ]:
def disguide_bypass_common_extra() -> dict:
    """Shared DisGUIDE-style loop for D1/D2 (generator differs; optional flow-matching loss)."""
    return {
        "engine": "disguide",
        "ensemble_size": 2,
        "latent_dim": 256,
        "g_iter": 1,
        "d_iter": 1,
        "rep_iter": 3,
        "replay_size": 100_000,
        "lambda_div": 0.2,
        "loss": "l1",
        "epoch_itrs": 150,
        "scheduler": "multistep",
        "steps": [0.4, 0.8],
        "scale": 0.3,
        "lr_student": 0.03,
        "lr_generator": 1e-4,
        "velocity_regularizer_weight": 0.0,
        "disable_pbar": False,
    }


def d1_extra() -> dict:
    return dict(disguide_bypass_common_extra())


def d2_extra() -> dict:
    return {
        **disguide_bypass_common_extra(),
        "procedural_grid_size": 4,
        "procedural_fourier_modes": 4,
        "procedural_blobs": 6,
    }


def disguide_extra() -> dict:
    """Hyperparameters aligned with disguide-main (GeneratorA + tanh, pre-transform)."""
    return {
        "ensemble_size": 2,
        "latent_dim": 256,
        "g_iter": 1,
        "d_iter": 1,
        "rep_iter": 3,
        "replay_size": 100_000,
        "lambda_div": 0.2,
        "loss": "l1",
        "epoch_itrs": 150,
        "scheduler": "multistep",
        "steps": [0.4, 0.8],
        "scale": 0.3,
        "lr_student": 0.03,
        "lr_generator": 1e-4,
        "grayscale": 0,
        "image_size": 32,
        "input_space": "pre-transform",
        "disable_pbar": False,
    }


@dataclass(frozen=True, slots=True)
class AttackJob:
    key: str
    label: str
    attack_kind: AttackKind
    attack_mode: str
    attack_extra: dict


ATTACK_JOBS = [
    AttackJob(
        key="d1_latent_manifold",
        label="D1 Latent-Manifold",
        attack_kind=AttackKind.LATENT_MANIFOLD,
        attack_mode="latent_manifold",
        attack_extra=d1_extra(),
    ),
    AttackJob(
        key="d2_procedural_natural",
        label="D2 Procedural-Natural",
        attack_kind=AttackKind.PROCEDURAL_NATURAL,
        attack_mode="procedural_natural",
        attack_extra=d2_extra(),
    ),
    AttackJob(
        key="disguide",
        label="DisGUIDE",
        attack_kind=AttackKind.DISGUIDE,
        attack_mode="disguide",
        attack_extra=disguide_extra(),
    ),
]

attack_results: dict[str, object] = {}

for job in ATTACK_JOBS:
    batch_size = DISGUIDE_BATCH_SIZE
    output_dir = RUN_ROOT / job.key
    if REUSE_ATTACK_RUNS and (output_dir / "experiment_result.json").exists():
        print(f"\n=== Skipping {job.label} (existing run at {output_dir}) ===")
        continue

    substitute_arch = "lenet5" if job.attack_kind == AttackKind.DISGUIDE else ARCHITECTURE
    spec = build_attack_spec(
        name=f"mnist-steal-{job.key}",
        attack_kind=job.attack_kind,
        attack_mode=job.attack_mode,
        attack_extra=job.attack_extra,
        attack_batch_size=batch_size,
        substitute_architecture=substitute_arch,
    )
    print(f"\n=== Running {job.label} -> {output_dir} ===")
    print(f"    query_budget={QUERY_BUDGET:,}, substitute_epochs={SUBSTITUTE_EPOCHS}")
    attack_results[job.key] = run_experiment(spec, output_dir=output_dir)
    metrics = attack_results[job.key].evaluation_summary.metrics
    print(
        f"{job.label}: queries={attack_results[job.key].query_summary.total_queries}, "
        f"accuracy={metrics.get('accuracy', float('nan')):.4f}, "
        f"fidelity={metrics.get('fidelity', float('nan')):.4f}"
    )

## Why ~10% accuracy? (diagnostics)

Data-free attacks on MNIST often fail for reasons **unrelated to query budget**:

1. **Generator collapse** — synthetic queries become near-constant tensors (pixel std ≈ 0). The victim then returns the same class for almost all queries.
2. **Label collapse** — the transfer set is dominated by one digit (e.g. 72% class 8 for DisGUIDE, 100% class 8 for D1).
3. **Domain gap** — the substitute/ensemble is trained on synthetic OOD images but evaluated on real MNIST test digits. DisGUIDE's in-attack `surrogate_accuracy` on real test data was already ~9.7%.
4. **DisGUIDE generator objective** — the generator is trained with *ensemble disagreement*, not victim feedback, so it has no incentive to produce digit-like inputs.

A **naive transfer-set attack** with real MNIST images reaches ~98% at 20k queries on the same victim — the pipeline works; the synthetic queries do not.

Run the cell below to inspect your transfer sets.

In [ ]:
import json
from collections import Counter


def diagnose_transferset(job_key: str, *, sample_size: int = 2000) -> dict:
    path = RUN_ROOT / job_key / "attack" / "transferset.pickle"
    if not path.exists():
        return {"attack": job_key, "error": f"missing {path}"}

    rows = torch.load(path, map_location="cpu", weights_only=False)
    take = rows[: min(sample_size, len(rows))]
    xs, ys = zip(*take)
    arrays = [np.asarray(x, dtype=np.float32) for x in xs]
    pixel_std = float(np.mean([arr.std() for arr in arrays]))
    pixel_range = (
        float(np.min([arr.min() for arr in arrays])),
        float(np.max([arr.max() for arr in arrays])),
    )
    argmax_labels = []
    entropies = []
    for y in ys:
        probs = torch.as_tensor(y, dtype=torch.float32)
        if probs.ndim == 0:
            argmax_labels.append(int(probs))
            continue
        argmax_labels.append(int(probs.argmax()))
        p = probs.clamp_min(1e-8)
        entropies.append(float((-p * p.log()).sum()))
    top = Counter(argmax_labels).most_common(3)
    dominant_frac = top[0][1] / len(argmax_labels) if top else 0.0

    disguide_sur_acc = None
    metrics_csv = RUN_ROOT / job_key / "attack" / "metrics.csv"
    if metrics_csv.exists():
        metrics_df = pd.read_csv(metrics_csv)
        if not metrics_df.empty and "surrogate_accuracy" in metrics_df.columns:
            disguide_sur_acc = float(metrics_df["surrogate_accuracy"].iloc[-1])

    return {
        "attack": job_key,
        "transfer_samples": len(rows),
        "pixel_std": pixel_std,
        "pixel_range": pixel_range,
        "dominant_label": top[0] if top else None,
        "dominant_label_fraction": dominant_frac,
        "label_entropy_mean": float(np.mean(entropies)) if entropies else None,
        "disguide_in_attack_sur_acc": disguide_sur_acc,
    }


diag_rows = [diagnose_transferset(job.key) for job in ATTACK_JOBS]
display(pd.DataFrame(diag_rows))

# Visual sanity check: first transfer-set image vs a real MNIST digit
fig, axes = plt.subplots(1, 2, figsize=(3.2, 1.8))
probe_path = RUN_ROOT / "d2_procedural_natural" / "attack" / "transferset.pickle"
if probe_path.exists():
    synth_x, _ = torch.load(probe_path, map_location="cpu", weights_only=False)[0]
    axes[0].imshow(to_mnist_image_array(synth_x), cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("D2 query (example)")
else:
    axes[0].set_title("no transfer set")
axes[0].axis("off")
real_x, real_y = testset[0]
axes[1].imshow(to_mnist_image_array(real_x), cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"real MNIST ({real_y})")
axes[1].axis("off")
fig.tight_layout()
plt.show()

## Results summary

In [ ]:
rows = []
for job in ATTACK_JOBS:
    result = attack_results[job.key]
    metrics = result.evaluation_summary.metrics
    rows.append(
        {
            "attack": job.label,
            "queries": result.query_summary.total_queries,
            "transfer_samples": result.attack_summary.metadata.get("num_samples"),
            "substitute_accuracy": metrics.get("accuracy"),
            "fidelity": metrics.get("fidelity"),
            "joint_accuracy": metrics.get("joint_accuracy"),
            "output_dir": str(RUN_ROOT / job.key),
        }
    )

summary = pd.DataFrame(rows).sort_values("substitute_accuracy", ascending=False)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2))
axes[0].bar(summary["attack"], summary["substitute_accuracy"], color="#0072B2")
axes[0].set_ylabel("Substitute test accuracy")
axes[0].set_ylim(0.0, 1.0)
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(summary["attack"], summary["fidelity"], color="#009E73")
axes[1].set_ylabel("Fidelity to victim")
axes[1].set_ylim(0.0, 1.0)
axes[1].tick_params(axis="x", rotation=20)

fig.suptitle("MNIST substitute models after data-free extraction", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mnist_stealing_summary.pdf", bbox_inches="tight")
plt.show()